In [ ]:
# Cell 1: Setup and Imports for Section 3.3
import pandas as pd
import numpy as np
import yfinance as yf
import wrds
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
from sqlalchemy import text
import getpass
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

print("="*80)
print("SECTION 3.3: PRICE ADJUSTMENTS & CORPORATE ACTIONS")
print("="*80 + "\n")

print("This section demonstrates:")
print("  1. How data providers handle stock splits")
print("  2. How data providers handle cash dividends")
print("  3. Impact on return calculations (raw vs adjusted)")
print("  4. Why adjusted prices are essential for analysis")
print("\n")

In [ ]:
# Cell 2: Connect to WRDS Database
print("="*80)
print("CONNECTING TO WRDS")
print("="*80 + "\n")

print("Please enter your WRDS credentials:")
print("(Your password will be hidden for security)\n")

username = input("Enter WRDS username: ")
password = getpass.getpass("Enter WRDS password: ")

try:
    print("\nConnecting to WRDS...")
    db = wrds.Connection(wrds_username=username, wrds_password=password)
    print("✓ Successfully connected to WRDS!")
    print(f"  Username: {username}")
    print(f"  Connection established\n")
except Exception as e:
    print(f"✗ Connection failed: {e}")
    print("\nPlease check:")
    print("  1. Your WRDS username and password are correct")
    print("  2. Your WRDS account is active")
    print("  3. You have accepted the WRDS data agreement")
    raise

print("="*80)
print("\n")

In [ ]:
# Cell 3: Stock Split Analysis - Apple's 4-for-1 Split (August 31, 2020)
print("="*80)
print("PART A: STOCK SPLIT ANALYSIS - APPLE 4-FOR-1 SPLIT")
print("="*80 + "\n")

print("Apple executed a 4-for-1 stock split on August 31, 2020")
print("  - Before split: ~$500/share")
print("  - After split: ~$125/share")
print("  - Shareholder with 1 share @ $500 now has 4 shares @ $125 each")
print("  - Total value unchanged: $500 = 4 × $125")
print("\n")

# Define split event
split_date = '2020-08-31'
start_date = '2020-07-01'
end_date = '2020-10-31'

# Fetch data from Yahoo Finance
print("Step 1: Fetching Apple data from Yahoo Finance...")
ticker = yf.Ticker('AAPL')

# Get historical data with auto_adjust=False to see raw prices
hist_raw = ticker.history(start=start_date, end=end_date, auto_adjust=False)
hist_raw['Date'] = hist_raw.index

# Get adjusted data
hist_adj = ticker.history(start=start_date, end=end_date, auto_adjust=True)
hist_adj['Date'] = hist_adj.index

print(f"  Retrieved {len(hist_raw)} trading days from Yahoo Finance")
print(f"  Split date: {split_date}")
print("\n")

# Display raw vs adjusted around split
split_window = hist_raw[(hist_raw.index >= '2020-08-25') & (hist_raw.index <= '2020-09-04')]
print("Yahoo Finance Data Around Split Date:")
print("-" * 80)
print(f"{'Date':<12} {'Raw Close':>12} {'Adj Close':>12} {'Raw/Adj Ratio':>15}")
print("-" * 80)

for idx, row in split_window.iterrows():
    adj_close = hist_adj.loc[idx, 'Close']
    ratio = row['Close'] / adj_close if adj_close != 0 else 0
    date_str = idx.strftime('%Y-%m-%d')
    print(f"{date_str:<12} ${row['Close']:>11.2f} ${adj_close:>11.2f} {ratio:>14.2f}x")

print("-" * 80)
print(f"\nObservations:")
print(f"  - Before Aug 31: Raw and Adjusted prices differ by ~4x (split factor)")
print(f"  - After Aug 31: Raw price drops by 4x (actual market price)")
print(f"  - Adjusted prices remain continuous (no artificial jump)")
print("\n")

In [ ]:
# Cell 4: Fetch CRSP Data for Apple Split (FIXED)
print("="*80)
print("STEP 2: FETCHING CRSP DATA FOR APPLE SPLIT")
print("="*80 + "\n")

# Get Apple's PERMNO
apple_permno_query = """
SELECT DISTINCT permno, ticker, comnam, namedt, nameendt
FROM crsp.dsenames
WHERE ticker = 'AAPL'
  AND namedt <= '2020-10-31'
  AND nameendt >= '2020-07-01'
ORDER BY namedt DESC
"""

print("Getting Apple PERMNO from CRSP...")
try:
    result = db.connection.execute(text(apple_permno_query))
    permno_data = pd.DataFrame(result.fetchall(), columns=result.keys())
    
    if len(permno_data) > 0:
        print("✓ Found Apple in CRSP:")
        display(permno_data)
        
        apple_permno = permno_data.iloc[0]['permno']
        print(f"\n  Using PERMNO: {apple_permno}")
        print("\n")
    else:
        print("✗ No PERMNO found for AAPL in specified date range")
        raise ValueError("Apple PERMNO not found")
        
except Exception as e:
    print(f"✗ Error fetching PERMNO: {e}")
    raise

# Fetch CRSP data around split
crsp_query = f"""
SELECT date, prc, cfacpr, cfacshr, ret, retx, vol
FROM crsp.dsf
WHERE permno = {apple_permno}
  AND date >= '2020-07-01'
  AND date <= '2020-10-31'
ORDER BY date
"""

print("Fetching CRSP daily stock file data...")
try:
    result = db.connection.execute(text(crsp_query))
    crsp_data = pd.DataFrame(result.fetchall(), columns=result.keys())
    
    if len(crsp_data) > 0:
        crsp_data['date'] = pd.to_datetime(crsp_data['date'])
        crsp_data.set_index('date', inplace=True)
        
        # Convert Decimal types to float (FIXED)
        crsp_data['prc'] = crsp_data['prc'].astype(float)
        crsp_data['cfacpr'] = crsp_data['cfacpr'].astype(float)
        crsp_data['cfacshr'] = crsp_data['cfacshr'].astype(float)
        crsp_data['ret'] = crsp_data['ret'].astype(float)
        crsp_data['retx'] = crsp_data['retx'].astype(float)
        crsp_data['vol'] = crsp_data['vol'].astype(float)
        
        # Calculate adjusted price using CRSP factors
        crsp_data['raw_price'] = abs(crsp_data['prc'])  # CRSP uses negative for bid/ask average
        crsp_data['adj_price'] = crsp_data['raw_price'] / crsp_data['cfacpr']
        
        print(f"✓ Retrieved {len(crsp_data)} trading days from CRSP")
        print("\n")
    else:
        print("✗ No CRSP data retrieved")
        raise ValueError("No CRSP data found")
        
except Exception as e:
    print(f"✗ Error fetching CRSP data: {e}")
    raise

# Display CRSP data around split
split_window_crsp = crsp_data[(crsp_data.index >= '2020-08-25') & (crsp_data.index <= '2020-09-04')]
print("CRSP Data Around Split Date:")
print("-" * 100)
print(f"{'Date':<12} {'Raw Price':>12} {'CFACPR':>12} {'Adj Price':>12} {'CFACSHR':>12} {'RET':>10}")
print("-" * 100)

for idx, row in split_window_crsp.iterrows():
    date_str = idx.strftime('%Y-%m-%d')
    ret_str = f"{row['ret']*100:.2f}%" if pd.notna(row['ret']) else "N/A"
    print(f"{date_str:<12} ${row['raw_price']:>11.2f} {row['cfacpr']:>11.4f} "
          f"${row['adj_price']:>11.2f} {row['cfacshr']:>11.4f} {ret_str:>9}")

print("-" * 100)
print("\nKey CRSP Variables:")
print("  - PRC: Daily closing price (negative = bid/ask average)")
print("  - CFACPR: Cumulative Factor to Adjust Price (accounts for splits & dividends)")
print("  - CFACSHR: Cumulative Factor to Adjust Shares (accounts for splits only)")
print("  - RET: Total return including dividends")
print("  - RETX: Return excluding dividends")
print(f"  - Notice: CFACPR changes by ~4x around split date")
print("\n")

In [ ]:
# Cell 5: Visualize Stock Split Impact
print("="*80)
print("VISUALIZATION: STOCK SPLIT IMPACT ON PRICES")
print("="*80 + "\n")

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('Apple 4-for-1 Stock Split (August 31, 2020)\nRaw vs Adjusted Prices', 
             fontsize=16, fontweight='bold')

# Plot 1: Yahoo Finance - Raw vs Adjusted
ax = axes[0, 0]
ax.plot(hist_raw.index, hist_raw['Close'], marker='o', linewidth=2, 
        label='Raw Price', color='red', markersize=4)
ax.plot(hist_adj.index, hist_adj['Close'], marker='s', linewidth=2, 
        label='Adjusted Price', color='blue', markersize=4, alpha=0.7)
ax.axvline(x=pd.to_datetime(split_date), color='green', linestyle='--', 
           linewidth=3, label='Split Date', alpha=0.7)

ax.set_title('Yahoo Finance: Raw vs Adjusted Prices', fontsize=13, fontweight='bold')
ax.set_xlabel('Date', fontweight='bold')
ax.set_ylabel('Price ($)', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

# Annotate the discontinuity
ax.annotate('Split causes 4x\nprice drop in raw series', 
            xy=(pd.to_datetime('2020-08-31'), 130), 
            xytext=(pd.to_datetime('2020-09-15'), 300),
            fontsize=10, ha='left',
            bbox=dict(boxstyle='round,pad=0.5', facecolor='yellow', alpha=0.7),
            arrowprops=dict(arrowstyle='->', lw=2, color='red'))

ax.annotate('Adjusted series\nremains continuous', 
            xy=(pd.to_datetime('2020-08-31'), 125), 
            xytext=(pd.to_datetime('2020-07-15'), 80),
            fontsize=10, ha='left',
            bbox=dict(boxstyle='round,pad=0.5', facecolor='lightblue', alpha=0.7),
            arrowprops=dict(arrowstyle='->', lw=2, color='blue'))

# Plot 2: CRSP - Raw vs Adjusted
ax = axes[0, 1]
ax.plot(crsp_data.index, crsp_data['raw_price'], marker='o', linewidth=2, 
        label='Raw Price (PRC)', color='red', markersize=4)
ax.plot(crsp_data.index, crsp_data['adj_price'], marker='s', linewidth=2, 
        label='Adjusted Price', color='orange', markersize=4, alpha=0.7)
ax.axvline(x=pd.to_datetime(split_date), color='green', linestyle='--', 
           linewidth=3, label='Split Date', alpha=0.7)

ax.set_title('CRSP: Raw vs Adjusted Prices', fontsize=13, fontweight='bold')
ax.set_xlabel('Date', fontweight='bold')
ax.set_ylabel('Price ($)', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

# Plot 3: CRSP Adjustment Factors
ax = axes[1, 0]
ax.plot(crsp_data.index, crsp_data['cfacpr'], marker='o', linewidth=2.5, 
        label='CFACPR (Price Factor)', color='purple', markersize=5)
ax.plot(crsp_data.index, crsp_data['cfacshr'], marker='s', linewidth=2.5, 
        label='CFACSHR (Share Factor)', color='brown', markersize=5, alpha=0.7)
ax.axvline(x=pd.to_datetime(split_date), color='green', linestyle='--', 
           linewidth=3, label='Split Date', alpha=0.7)

ax.set_title('CRSP Adjustment Factors Over Time', fontsize=13, fontweight='bold')
ax.set_xlabel('Date', fontweight='bold')
ax.set_ylabel('Cumulative Factor', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

# Add annotation showing 4x change
pre_split_cfacpr = crsp_data.loc[crsp_data.index < '2020-08-31', 'cfacpr'].iloc[-1]
post_split_cfacpr = crsp_data.loc[crsp_data.index > '2020-08-31', 'cfacpr'].iloc[0]
factor_change = post_split_cfacpr / pre_split_cfacpr

ax.text(0.05, 0.95, f'Factor Change: {factor_change:.2f}x\n(4-for-1 split)', 
        transform=ax.transAxes, fontsize=11, verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.7), fontweight='bold')

# Plot 4: Mathematical Demonstration
ax = axes[1, 1]
ax.axis('off')

# Create text explanation
explanation = f"""
MATHEMATICAL DEMONSTRATION OF STOCK SPLIT

Split Ratio: 4-for-1 (each share becomes 4 shares)

Before Split (Aug 28, 2020):
  Raw Price: ${hist_raw.loc['2020-08-28', 'Close']:.2f}
  Shares Held: 1
  Total Value: ${hist_raw.loc['2020-08-28', 'Close']:.2f}

After Split (Sep 1, 2020):
  Raw Price: ${hist_raw.loc['2020-09-01', 'Close']:.2f}
  Shares Held: 4
  Total Value: 4 × ${hist_raw.loc['2020-09-01', 'Close']:.2f} = ${4 * hist_raw.loc['2020-09-01', 'Close']:.2f}

Verification:
  Value Change: ${4 * hist_raw.loc['2020-09-01', 'Close'] - hist_raw.loc['2020-08-28', 'Close']:.2f}
  (Close to zero = wealth preserved)

Adjusted Price Calculation (Yahoo):
  Adj Close = Raw Close / Split Factor
  Pre-split adj: ${hist_raw.loc['2020-08-28', 'Close']:.2f} / 4 = ${hist_adj.loc['2020-08-28', 'Close']:.2f}
  Post-split adj: ${hist_raw.loc['2020-09-01', 'Close']:.2f} / 1 = ${hist_adj.loc['2020-09-01', 'Close']:.2f}

CRSP Adjustment (using CFACPR):
  Adj Price = Raw Price / CFACPR
  CFACPR changes from {pre_split_cfacpr:.4f} to {post_split_cfacpr:.4f}
  This ensures continuity in adjusted price series
"""

ax.text(0.1, 0.95, explanation, transform=ax.transAxes, fontsize=10,
        verticalalignment='top', family='monospace',
        bbox=dict(boxstyle='round', facecolor='lightgray', alpha=0.8))

plt.tight_layout()
plt.savefig('/home/aditya/FE511_Project/section_3_3_stock_split.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Visualization saved to: /home/aditya/FE511_Project/section_3_3_stock_split.png")
print("\n")

In [ ]:
# Cell 6: Dividend Adjustment Analysis
print("="*80)
print("PART B: CASH DIVIDEND ADJUSTMENT ANALYSIS")
print("="*80 + "\n")

print("Analyzing how dividends affect prices and returns")
print("Example: If stock closes at $100 and pays $1 dividend:")
print("  - Next day opening price typically drops to ~$99 (ex-dividend)")
print("  - Raw price series shows discontinuity")
print("  - Adjusted price series accounts for dividend to show true return")
print("\n")

# Use Apple data from 2024 (recent dividends)
div_start = '2024-01-01'
div_end = '2024-12-31'

print("Fetching Apple dividend data from Yahoo Finance...")
ticker_div = yf.Ticker('AAPL')
hist_div_raw = ticker_div.history(start=div_start, end=div_end, auto_adjust=False)
hist_div_adj = ticker_div.history(start=div_start, end=div_end, auto_adjust=True)

# Get dividend events
dividends = ticker_div.dividends
dividends_2024 = dividends[(dividends.index >= div_start) & (dividends.index <= div_end)]

print(f"  Retrieved {len(hist_div_raw)} trading days")
print(f"  Found {len(dividends_2024)} dividend payments in 2024")
print("\n")

if len(dividends_2024) > 0:
    print("Dividend Payment Dates and Amounts:")
    print("-" * 50)
    for date, amount in dividends_2024.items():
        print(f"  {date.strftime('%Y-%m-%d')}: ${amount:.4f}")
    print("\n")
    
    # Analyze first dividend event
    first_div_date = dividends_2024.index[0]
    div_amount = dividends_2024.iloc[0]
    
    # Get prices around dividend date
    window_start = first_div_date - timedelta(days=5)
    window_end = first_div_date + timedelta(days=5)
    
    div_window_raw = hist_div_raw[(hist_div_raw.index >= window_start) & 
                                   (hist_div_raw.index <= window_end)]
    div_window_adj = hist_div_adj[(hist_div_adj.index >= window_start) & 
                                   (hist_div_adj.index <= window_end)]
    
    print(f"Analysis of Dividend Event: {first_div_date.strftime('%Y-%m-%d')}")
    print(f"Dividend Amount: ${div_amount:.4f}")
    print("-" * 80)
    print(f"{'Date':<12} {'Raw Close':>12} {'Adj Close':>12} {'Difference':>12} {'Dividend':>12}")
    print("-" * 80)
    
    for idx, row in div_window_raw.iterrows():
        adj_close = div_window_adj.loc[idx, 'Close']
        diff = row['Close'] - adj_close
        div_str = f"${div_amount:.4f}" if idx == first_div_date else "-"
        date_str = idx.strftime('%Y-%m-%d')
        print(f"{date_str:<12} ${row['Close']:>11.2f} ${adj_close:>11.2f} "
              f"${diff:>11.2f} {div_str:>11}")
    
    print("-" * 80)
    print("\n")
else:
    print("⚠️  No dividends found in 2024 period.")
    print("   Using alternative period (2023) for demonstration...")
    div_start = '2023-01-01'
    div_end = '2023-12-31'
    hist_div_raw = ticker_div.history(start=div_start, end=div_end, auto_adjust=False)
    hist_div_adj = ticker_div.history(start=div_start, end=div_end, auto_adjust=True)
    dividends_2024 = dividends[(dividends.index >= div_start) & (dividends.index <= div_end)]
    print(f"   Found {len(dividends_2024)} dividend payments in 2023")
    print("\n")

In [ ]:
# Cell 7: Return Calculation - Raw vs Adjusted Prices
print("="*80)
print("IMPACT ON RETURN CALCULATIONS")
print("="*80 + "\n")

print("Demonstrating why adjusted prices MUST be used for return calculations")
print("\n")

# Select a period with a dividend
if len(dividends_2024) > 0:
    div_date = dividends_2024.index[0]
    div_amt = dividends_2024.iloc[0]
    
    # Get day before and day after dividend
    try:
        day_before = hist_div_raw.index[hist_div_raw.index < div_date][-1]
        day_after = hist_div_raw.index[hist_div_raw.index > div_date][0]
        
        # Raw prices
        raw_before = hist_div_raw.loc[day_before, 'Close']
        raw_after = hist_div_raw.loc[day_after, 'Close']
        
        # Adjusted prices
        adj_before = hist_div_adj.loc[day_before, 'Close']
        adj_after = hist_div_adj.loc[day_after, 'Close']
        
        # Calculate returns both ways
        print("Return Calculation Around Dividend Event:")
        print("=" * 80)
        print(f"Dividend Date: {div_date.strftime('%Y-%m-%d')}")
        print(f"Dividend Amount: ${div_amt:.4f}")
        print("\n")
        
        # Method 1: Using Raw Prices (INCORRECT)
        print("METHOD 1: Using Raw Prices (INCORRECT)")
        print("-" * 80)
        print(f"Day Before ({day_before.strftime('%Y-%m-%d')}): Price = ${raw_before:.2f}")
        print(f"Day After ({day_after.strftime('%Y-%m-%d')}):  Price = ${raw_after:.2f}")
        
        raw_return = (raw_after - raw_before) / raw_before
        print(f"\nReturn = (P_t - P_(t-1)) / P_(t-1)")
        print(f"       = (${raw_after:.2f} - ${raw_before:.2f}) / ${raw_before:.2f}")
        print(f"       = {raw_return:.4f} or {raw_return*100:.2f}%")
        print("\n⚠️  This is INCORRECT because it doesn't account for the dividend!")
        print(f"    Investor received ${div_amt:.4f} dividend but return shows negative!")
        print("\n")
        
        # Method 2: Using Raw Prices + Dividend (CORRECT but manual)
        print("METHOD 2: Using Raw Prices + Dividend (CORRECT but cumbersome)")
        print("-" * 80)
        total_return_manual = (raw_after + div_amt - raw_before) / raw_before
        print(f"Return = (P_t + D_t - P_(t-1)) / P_(t-1)")
        print(f"       = (${raw_after:.2f} + ${div_amt:.4f} - ${raw_before:.2f}) / ${raw_before:.2f}")
        print(f"       = {total_return_manual:.4f} or {total_return_manual*100:.2f}%")
        print("\n✓ This is CORRECT but requires tracking all dividend payments")
        print("\n")
        
        # Method 3: Using Adjusted Prices (CORRECT and automatic)
        print("METHOD 3: Using Adjusted Prices (CORRECT and automatic)")
        print("-" * 80)
        print(f"Day Before ({day_before.strftime('%Y-%m-%d')}): Adj Price = ${adj_before:.2f}")
        print(f"Day After ({day_after.strftime('%Y-%m-%d')}):  Adj Price = ${adj_after:.2f}")
        
        adj_return = (adj_after - adj_before) / adj_before
        print(f"\nReturn = (P_adj,t - P_adj,(t-1)) / P_adj,(t-1)")
        print(f"       = (${adj_after:.2f} - ${adj_before:.2f}) / ${adj_before:.2f}")
        print(f"       = {adj_return:.4f} or {adj_return*100:.2f}%")
        print("\n✓ This is CORRECT and dividend is automatically included!")
        print("  Adjusted price series already incorporates dividend payments")
        print("\n")
        
        # Comparison
        print("COMPARISON OF METHODS")
        print("=" * 80)
        print(f"Method 1 (Raw, no dividend):        {raw_return*100:>8.2f}%  ❌ WRONG")
        print(f"Method 2 (Raw + dividend manual):   {total_return_manual*100:>8.2f}%  ✓ Correct")
        print(f"Method 3 (Adjusted prices):         {adj_return*100:>8.2f}%  ✓ Correct")
        print(f"\nDifference between Method 1 and Method 3: {(adj_return - raw_return) * 100:.2f} percentage points")
        print("\n")
        
    except IndexError:
        print("⚠️  Unable to find trading days immediately around dividend date")
        print("\n")
else:
    print("⚠️  No dividend events available for analysis")
    print("\n")

In [ ]:
# Cell 8: Visualization - Dividend Impact
print("="*80)
print("VISUALIZATION: DIVIDEND IMPACT ON PRICES AND RETURNS")
print("="*80 + "\n")

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('Cash Dividend Impact: Apple Stock\nRaw vs Adjusted Price Series', 
             fontsize=16, fontweight='bold')

# Plot 1: Full year - Raw vs Adjusted
ax = axes[0, 0]
ax.plot(hist_div_raw.index, hist_div_raw['Close'], linewidth=2, 
        label='Raw Close', color='red', alpha=0.7)
ax.plot(hist_div_adj.index, hist_div_adj['Close'], linewidth=2, 
        label='Adjusted Close', color='blue', alpha=0.7)

# Mark dividend dates
for div_date in dividends_2024.index:
    ax.axvline(x=div_date, color='green', linestyle='--', linewidth=1, alpha=0.5)

ax.set_title(f'Apple {div_start[:4]}: Raw vs Adjusted Prices', fontsize=13, fontweight='bold')
ax.set_xlabel('Date', fontweight='bold')
ax.set_ylabel('Price ($)', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

# Add annotation
ax.text(0.02, 0.98, f'{len(dividends_2024)} dividend payments\n(green dashed lines)', 
        transform=ax.transAxes, fontsize=10, verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.6))

# Plot 2: Zoom in on first dividend
if len(dividends_2024) > 0:
    ax = axes[0, 1]
    div_date = dividends_2024.index[0]
    zoom_start = div_date - timedelta(days=10)
    zoom_end = div_date + timedelta(days=10)
    
    zoom_raw = hist_div_raw[(hist_div_raw.index >= zoom_start) & (hist_div_raw.index <= zoom_end)]
    zoom_adj = hist_div_adj[(hist_div_adj.index >= zoom_start) & (hist_div_adj.index <= zoom_end)]
    
    ax.plot(zoom_raw.index, zoom_raw['Close'], marker='o', linewidth=2.5, markersize=6,
            label='Raw Close', color='red')
    ax.plot(zoom_adj.index, zoom_adj['Close'], marker='s', linewidth=2.5, markersize=6,
            label='Adjusted Close', color='blue', alpha=0.7)
    ax.axvline(x=div_date, color='green', linestyle='--', linewidth=3, 
               label=f'Div: ${dividends_2024.iloc[0]:.4f}', alpha=0.7)
    
    ax.set_title(f'Dividend Event: {div_date.strftime("%Y-%m-%d")}', 
                 fontsize=13, fontweight='bold')
    ax.set_xlabel('Date', fontweight='bold')
    ax.set_ylabel('Price ($)', fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.tick_params(axis='x', rotation=45)

# Plot 3: Return comparison
ax = axes[1, 0]

# Calculate daily returns both ways
hist_div_raw['raw_return'] = hist_div_raw['Close'].pct_change()
hist_div_adj['adj_return'] = hist_div_adj['Close'].pct_change()

# Plot returns
ax.plot(hist_div_raw.index, hist_div_raw['raw_return'] * 100, 
        linewidth=1.5, label='Returns from Raw Prices', color='red', alpha=0.6)
ax.plot(hist_div_adj.index, hist_div_adj['adj_return'] * 100, 
        linewidth=1.5, label='Returns from Adjusted Prices', color='blue', alpha=0.6)

# Highlight dividend dates
for div_date in dividends_2024.index:
    ax.axvline(x=div_date, color='green', linestyle='--', linewidth=1, alpha=0.3)

ax.set_title('Daily Returns: Raw vs Adjusted Prices', fontsize=13, fontweight='bold')
ax.set_xlabel('Date', fontweight='bold')
ax.set_ylabel('Daily Return (%)', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_ylim(-5, 5)

# Plot 4: Cumulative returns
ax = axes[1, 1]

# Calculate cumulative returns
cum_ret_raw = (1 + hist_div_raw['raw_return']).cumprod() - 1
cum_ret_adj = (1 + hist_div_adj['adj_return']).cumprod() - 1

ax.plot(cum_ret_raw.index, cum_ret_raw * 100, linewidth=2.5, 
        label='Cumulative Return (Raw)', color='red')
ax.plot(cum_ret_adj.index, cum_ret_adj * 100, linewidth=2.5, 
        label='Cumulative Return (Adjusted)', color='blue')

ax.set_title(f'Cumulative Returns Comparison ({div_start[:4]})', fontsize=13, fontweight='bold')
ax.set_xlabel('Date', fontweight='bold')
ax.set_ylabel('Cumulative Return (%)', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

# Add final return difference
final_diff = (cum_ret_adj.iloc[-1] - cum_ret_raw.iloc[-1]) * 100
ax.text(0.02, 0.98, f'Difference: {final_diff:.2f}%\n(Due to dividends)', 
        transform=ax.transAxes, fontsize=11, verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.7), fontweight='bold')

plt.tight_layout()
plt.savefig('/home/aditya/FE511_Project/section_3_3_dividends.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Visualization saved to: /home/aditya/FE511_Project/section_3_3_dividends.png")
print("\n")

In [ ]:
# Cell 9: Summary Table and Conclusions
print("="*80)
print("SECTION 3.3 SUMMARY: WHY ADJUSTED PRICES ARE ESSENTIAL")
print("="*80 + "\n")

fig, ax = plt.subplots(figsize=(14, 8))
ax.axis('tight')
ax.axis('off')

summary_data = [
    ['Corporate Action', 'Effect on Raw Price', 'Effect on Adjusted Price', 'Why Adjustment Needed'],
    ['Stock Split\n(4-for-1)', 'Drops by 75%\n($500 → $125)', 'Remains continuous\n(retroactive adjustment)', 'Prevents artificial\n-75% return'],
    ['Cash Dividend\n($1.00)', 'Drops by ~$1\non ex-date', 'Smooth continuation\n(incorporates dividend)', 'Captures total return\n(price + dividend)'],
    ['Stock Dividend\n(10%)', 'Drops by ~10%', 'Adjusted for extra shares', 'Reflects true value\nper share'],
    ['Spin-off', 'Drops by spinoff value', 'Allocated appropriately', 'Maintains accurate\nperformance'],
]

cell_colors = [['#404040']*4]  # Header row
for i in range(1, len(summary_data)):
    cell_colors.append(['#E8F4F8', '#FFE4E1', '#E1FFE1', '#FFF9E1'])

table = ax.table(cellText=summary_data, cellColours=cell_colors,
                 cellLoc='left', loc='center',
                 colWidths=[0.15, 0.25, 0.25, 0.25])

table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1, 3)

# Style header
for i in range(4):
    cell = table[(0, i)]
    cell.set_text_props(weight='bold', color='white', size=11)

ax.set_title('Section 3.3: Corporate Actions and Price Adjustments Summary\n\n' + 
             'Key Principle: Always use ADJUSTED prices for return calculations and backtesting',
             fontsize=14, fontweight='bold', pad=20)

plt.savefig('/home/aditya/FE511_Project/section_3_3_summary.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Summary table saved to: /home/aditya/FE511_Project/section_3_3_summary.png")
print("\n")

print("="*80)
print("KEY TAKEAWAYS FROM SECTION 3.3")
print("="*80)
print("""
1. STOCK SPLITS
   - Raw prices show discontinuity (sudden drop)
   - Adjusted prices remain continuous
   - Both Yahoo and CRSP handle splits correctly
   - CRSP uses CFACPR factor, Yahoo uses split ratio

2. DIVIDENDS
   - Raw prices drop on ex-dividend date
   - Using raw prices UNDERESTIMATES returns
   - Adjusted prices incorporate dividend payments automatically
   - Essential for accurate total return calculation

3. RETURN CALCULATION FORMULA
   
   WRONG: R = (P_t - P_(t-1)) / P_(t-1)    [Using raw prices]
   
   CORRECT: R = (P_t + D_t - P_(t-1)) / P_(t-1)    [Manual approach]
   
   BEST: R = (P_adj,t - P_adj,(t-1)) / P_adj,(t-1)    [Using adjusted prices]

4. WHY THIS MATTERS FOR PROJECT
   - Part 4 backtest REQUIRES adjusted prices
   - Without adjustments, strategy returns would be inaccurate
   - Data providers must handle adjustments consistently
   - Historical accuracy depends on proper adjustment factors

5. PROVIDER DIFFERENCES
   - Yahoo: Provides 'Adj Close' field automatically
   - CRSP: Provides CFACPR factor for manual adjustment
   - Compustat: Uses AJEXDI (adjustment factor)
   - All three implement same economic principle differently

6. BEST PRACTICES
   ✓ Always use adjusted prices for return calculations
   ✓ Always use adjusted prices for backtesting
   ✓ Always use adjusted prices for performance analysis
   ✓ Verify adjustment methodology matches across sources
   ✓ Understand each provider's adjustment factors
""")

print("="*80)
print("SECTION 3.3 COMPLETE")
print("="*80)
print("\n✓ All visualizations saved to /home/aditya/FE511_Project/")
print("  - section_3_3_stock_split.png")
print("  - section_3_3_dividends.png")
print("  - section_3_3_summary.png")